# Oil Price Forecasting: When the Model Doesn't Know What It Doesn't Know

**Agentic Forecasting Bootcamp — Information Session Demo**

---

Imagine you're the forecasting analyst at a mid-sized logistics firm. Every day you generate a 14-day ahead forecast of WTI crude oil prices to inform fuel hedging decisions. You've been doing this since January 2025.

Your model is Prophet — a production-grade statistical forecasting library developed at Meta. It captures trend and seasonality from historical price data and produces calibrated 95% confidence intervals. Through 2025 it looked solid. Then came 2026.

This notebook walks through that story.

In [9]:
from __future__ import annotations

import datetime
import logging
import os
import warnings
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from prophet import Prophet

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd  # fallback: use CWD
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

PRICE_CACHE = DATA_DIR / "wti_price_history.parquet"
FORECAST_CACHE = DATA_DIR / "energy_case_study_forecasts.parquet"

# ── Simulation constants ───────────────────────────────────────────────────────
SIMULATION_START = pd.Timestamp("2025-01-01")
# Run through mid-April 2026 so the biggest March misses resolve inside the animation
SIMULATION_END = pd.Timestamp("2026-04-15")
HORIZON = 14  # calendar days ahead
CI_WIDTH = 0.95

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir: {DATA_DIR}")
print("Setup complete.")

Repo root: /Users/ethanjackson/agentic-forecasting
Data dir: /Users/ethanjackson/agentic-forecasting/data
Setup complete.


## Load WTI price data

We use Yahoo Finance's `CL=F` — the WTI crude oil continuous front-month futures contract.
It tracks the spot price within cents and requires no API key.
Data runs from January 2021 through today.

> **Note on data source**: `CL=F` is a futures price, not the EIA-posted spot price (`DCOILWTICO` on FRED).
> For daily oil price analysis these two series are virtually indistinguishable — the front-month
> futures contract converges to spot at expiry. Using `CL=F` is also a natural bridge into Act 4,
> where we discuss what a full futures *curve* would add.

In [10]:
def load_wti_prices(cache_path: Path) -> pd.DataFrame:
    """Load WTI front-month prices, using a local parquet cache when available."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        print(f"Loaded from cache: {df.index[0].date()} → {df.index[-1].date()} ({len(df):,} trading days)")
        return df

    print("Downloading WTI data from Yahoo Finance...")
    raw = yf.download("CL=F", start="2021-01-01", progress=False, auto_adjust=True)
    # Flatten MultiIndex columns if present
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df = raw[["Close"]].rename(columns={"Close": "price"})
    # Normalise index to date-only timestamps
    df.index = pd.DatetimeIndex([pd.Timestamp(str(d)[:10]) for d in df.index])
    df.index.name = "date"
    df = df.dropna().sort_index()
    df.to_parquet(cache_path)
    print(f"Downloaded & cached: {df.index[0].date()} → {df.index[-1].date()} ({len(df):,} trading days)")
    return df


price_df = load_wti_prices(PRICE_CACHE)
print(f"Latest price: ${price_df['price'].iloc[-1]:.2f} on {price_df.index[-1].date()}")

Loaded from cache: 2021-01-04 → 2026-04-30 (1,339 trading days)
Latest price: $104.35 on 2026-04-30


---

## Act 1 — The World Before the Simulation

Before we run a single forecast, let's ground ourselves in the full price history.
Oil markets are not stationary — they're shaped by geopolitics, macroeconomics, and supply
decisions that no historical pattern can fully anticipate.

In [11]:
def make_context_chart(price_df: pd.DataFrame) -> go.Figure:
    """Annotated WTI price history from 2021 through start of simulation."""
    history = price_df.loc[:"2024-12-31"]
    sim_era = price_df.loc["2025-01-01":]

    events: list[tuple[str, str, str]] = [
        ("2021-10-26", "Demand recovery peak", "above"),
        ("2022-02-24", "Russia invades Ukraine", "above"),
        ("2022-06-08", "~$120 peak", "above"),
        ("2022-09-05", "OPEC+ output cuts", "below"),
        ("2023-09-05", "Saudi Arabia extends cuts", "above"),
        ("2024-01-15", "Red Sea disruptions", "below"),
        ("2024-09-03", "Price softens", "above"),
    ]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=history.index, y=history["price"],
        mode="lines",
        line=dict(color="#6baed6", width=2),
        name="WTI Price (2021–2024)",
    ))

    fig.add_trace(go.Scatter(
        x=sim_era.index, y=sim_era["price"],
        mode="lines",
        line=dict(color="#e6550d", width=2),
        name="WTI Price (2025–present)",
    ))

    fig.add_vline(
        x=SIMULATION_START.timestamp() * 1000,
        line=dict(color="#e6550d", width=2, dash="dash"),
        annotation_text="Simulation begins →",
        annotation_position="top left",
        annotation_font=dict(size=12, color="#e6550d"),
    )

    for date_str, label, position in events:
        ts = pd.Timestamp(date_str)
        nearby = price_df.index[price_df.index >= ts]
        if len(nearby) == 0:
            continue
        ts = nearby[0]
        price_val = float(price_df.loc[ts, "price"])
        ay = -45 if position == "above" else 45
        fig.add_annotation(
            x=ts, y=price_val,
            text=label,
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=1.5,
            arrowcolor="#636363",
            ax=0, ay=ay,
            font=dict(size=10, color="#252525"),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#bdbdbd",
            borderwidth=1,
            borderpad=3,
        )

    fig.update_layout(
        title=dict(text="WTI Crude Oil Price: 2021–Present", font=dict(size=20)),
        xaxis=dict(title="Date", showgrid=True, gridcolor="#f0f0f0"),
        yaxis=dict(title="Price (USD/bbl)", showgrid=True, gridcolor="#f0f0f0"),
        template="plotly_white",
        legend=dict(orientation="h", y=-0.15),
        height=500,
        margin=dict(t=70, b=80, l=60, r=40),
    )
    return fig


make_context_chart(price_df).show()

---

## Pre-compute: Rolling 14-Day Prophet Forecasts

Starting January 1, 2025, we simulate what it would have looked like to run a daily
14-day-ahead forecast using Prophet:

- **Monthly refits**: the model re-trains at the start of each calendar month on all data
  through the previous month-end. This reflects a realistic production cadence — you don't
  refit every day, but you do update regularly.
- **Forecast target**: the price 14 calendar days from today, resolved on the nearest
  available trading day.
- **Uncertainty interval**: 95% (`interval_width=0.95`).

Results are cached to `data/energy_case_study_forecasts.parquet`.
**First run: 2–4 minutes. Subsequent runs: instant.**

In [12]:
def find_nearest_trading_day(
    target: pd.Timestamp, index: pd.DatetimeIndex
) -> pd.Timestamp | None:
    """Return the nearest trading day on or after `target` within the index."""
    candidates = index[index >= target]
    return candidates[0] if len(candidates) > 0 else None


def compute_rolling_forecasts(
    price_df: pd.DataFrame,
    simulation_start: pd.Timestamp,
    simulation_end: pd.Timestamp,
    horizon_days: int,
    ci_width: float,
    cache_path: Path,
) -> pd.DataFrame:
    """Run monthly-refit Prophet forecasts for every simulation trading day.

    Returns a DataFrame with columns:
        sim_day, resolution_date, yhat, yhat_lower, yhat_upper, actual_price, inside_ci
    """
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df):,} pre-computed forecasts from cache.")
        return df

    sim_days = price_df.loc[simulation_start:simulation_end].index.tolist()
    print(f"Computing forecasts for {len(sim_days)} simulation days...")

    # Month-end cutoffs: train on data through each cutoff, use model for next month's sim days
    monthly_cutoffs: list[pd.Timestamp] = pd.date_range(
        start="2024-12-31",
        end=simulation_end + timedelta(days=31),
        freq="ME",
    ).tolist()

    def cutoff_for(day: pd.Timestamp) -> pd.Timestamp:
        """Return the most recent monthly cutoff strictly before `day`."""
        past = [c for c in monthly_cutoffs if c < day]
        return past[-1] if past else monthly_cutoffs[0]

    # Group sim_days by their cutoff so we train each model once
    cutoff_groups: dict[pd.Timestamp, list[pd.Timestamp]] = {}
    for sim_day in sim_days:
        cutoff = cutoff_for(sim_day)
        cutoff_groups.setdefault(cutoff, []).append(sim_day)

    results: list[dict] = []

    for cutoff, days_for_model in sorted(cutoff_groups.items()):
        print(f"  Fitting Prophet on data through {cutoff.date()}...", end=" ", flush=True)

        train_df = price_df.loc[:cutoff][["price"]].reset_index()
        train_df.columns = pd.Index(["ds", "y"])

        model = Prophet(
            interval_width=ci_width,
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
        )
        model.fit(train_df)

        # Generate enough future periods to cover ALL resolution dates for this group
        max_resolution_calendar = max(d + timedelta(days=horizon_days) for d in days_for_model)
        periods_needed = (max_resolution_calendar - cutoff).days + 5
        future = model.make_future_dataframe(periods=periods_needed, freq="D")
        pred = model.predict(future).set_index("ds")

        print("done")

        for sim_day in days_for_model:
            resolution_calendar = sim_day + timedelta(days=horizon_days)
            resolution_day = find_nearest_trading_day(resolution_calendar, price_df.index)
            if resolution_day is None:
                continue

            # Look up Prophet forecast at the calendar target date
            if resolution_calendar not in pred.index:
                continue

            row = pred.loc[resolution_calendar]
            actual_price = float(price_df.loc[resolution_day, "price"])
            inside_ci = bool(float(row["yhat_lower"]) <= actual_price <= float(row["yhat_upper"]))

            results.append({
                "sim_day": sim_day,
                "resolution_date": resolution_day,
                "yhat": float(row["yhat"]),
                "yhat_lower": float(row["yhat_lower"]),
                "yhat_upper": float(row["yhat_upper"]),
                "actual_price": actual_price,
                "inside_ci": inside_ci,
            })

    forecasts_df = pd.DataFrame(results)
    forecasts_df.to_parquet(cache_path)
    print(f"\nSaved {len(forecasts_df):,} forecast records to {cache_path}")
    return forecasts_df


forecasts_df = compute_rolling_forecasts(
    price_df=price_df,
    simulation_start=SIMULATION_START,
    simulation_end=SIMULATION_END,
    horizon_days=HORIZON,
    ci_width=CI_WIDTH,
    cache_path=FORECAST_CACHE,
)
print(f"\nForecast date range: {forecasts_df['sim_day'].min().date()} → {forecasts_df['sim_day'].max().date()}")
forecasts_df.head()

Loaded 323 pre-computed forecasts from cache.

Forecast date range: 2025-01-02 → 2026-04-15


,sim_day,resolution_date,yhat,yhat_lower,yhat_upper,actual_price,inside_ci
0,2025-01-02,2025-01-16,69.626297,62.381704,77.201834,78.680000,False
1,2025-01-03,2025-01-17,69.850408,62.033605,76.992983,77.879997,False
2,2025-01-06,2025-01-21,70.489526,62.968725,77.822238,75.889999,True
3,2025-01-07,2025-01-21,70.682736,63.021171,78.541807,75.889999,True
4,2025-01-08,2025-01-22,70.862220,63.829971,78.518670,75.440002,True


---

## Act 2 — The Rolling Forecast in Motion

The animation below replays the simulation day by day, from January 2025 through March 2026.

**How to read it:**
- The **blue line** is the realized WTI price — it reveals itself as time passes.
- Each **orange bar** is the 95% CI for one 14-day-ahead forecast, placed at its *resolution date* (14 days after the forecast was made). Bars accumulate as forecasts resolve.
- The **brighter orange bar** is the *current* forecast — its resolution date is still 14 days away.
- **Green dots** mark resolutions that landed inside the 95% CI. **Red ✕ marks** indicate misses — actual prices that fell outside the predicted range.
- The **scorecard** (top right) tracks running coverage.

Use **▶ Play** to run through 2025 at speed. Pause and step manually as you enter 2026.

In [ ]:
def build_forecast_animation(
    price_df: pd.DataFrame,
    forecasts_df: pd.DataFrame,
) -> go.Figure:
    """Build a Plotly animation showing accumulated 95% CI bars at resolution dates.

    Each bar is centred on the resolution date (sim_day + 14) and spans [yhat_lower,
    yhat_upper].  Adjacent bars share their boundaries (computed as midpoints between
    consecutive resolution dates) so there are no gaps or overlaps.  Hit/miss markers
    show where the actual price landed relative to each bar.
    """
    CLR_PRE_SIM      = "#9ecae1"
    CLR_REVEALED     = "#2171b5"
    CLR_CI_PAST_FILL = "rgba(253, 141, 60, 0.22)"
    CLR_CI_CURR_FILL = "rgba(253, 141, 60, 0.60)"
    CLR_DAY_LINE     = "rgba(120, 120, 120, 0.45)"
    CLR_HIT          = "#31a354"
    CLR_MISS         = "#de2d26"

    pre_sim      = price_df.loc[:"2024-12-31"]
    price_series = price_df["price"]
    y_min = float(price_series.min()) * 0.90
    y_max = float(price_series.max()) * 1.10

    sim_days: list[pd.Timestamp] = sorted(forecasts_df["sim_day"].unique().tolist())

    def _adjacent_bars(rows: pd.DataFrame) -> tuple[list, list]:
        """Build non-overlapping CI bars with boundaries at midpoints between dates.

        Each bar's left/right edge is the midpoint between its resolution date and
        its neighbours', so bars tile seamlessly with no gaps or overlapping borders.
        """
        if rows.empty:
            return [], []
        rows_s = rows.sort_values("resolution_date").reset_index(drop=True)
        dates = rows_s["resolution_date"].tolist()
        lows  = rows_s["yhat_lower"].tolist()
        highs = rows_s["yhat_upper"].tolist()
        n = len(dates)
        all_x: list = []
        all_y: list = []
        for i in range(n):
            left  = dates[i] - (dates[i] - dates[i - 1]) / 2 if i > 0     else dates[i] - (dates[1] - dates[0]) / 2 if n > 1 else dates[i] - pd.Timedelta(days=1)
            right = dates[i] + (dates[i + 1] - dates[i]) / 2 if i < n - 1 else dates[i] + (dates[-1] - dates[-2]) / 2 if n > 1 else dates[i] + pd.Timedelta(days=1)
            if all_x:
                all_x.append(None)
                all_y.append(None)
            all_x.extend([left, right, right, left, left])
            all_y.extend([lows[i], lows[i], highs[i], highs[i], lows[i]])
        return all_x, all_y

    def _curr_bar_coords(
        fc: pd.Series,
        past_res: pd.DataFrame,
    ) -> tuple[list, list]:
        """CI bar for the current (not-yet-resolved) forecast.

        Width is derived from the spacing between the last two resolved bars so
        it stays visually consistent regardless of how far ahead the resolution
        date falls (avoids the jump when the first resolutions arrive).
        """
        res_date = fc["resolution_date"]
        lo       = float(fc["yhat_lower"])
        hi       = float(fc["yhat_upper"])
        if len(past_res) >= 2:
            last_two = past_res.sort_values("resolution_date")["resolution_date"].iloc[-2:]
            half_w   = (last_two.iloc[1] - last_two.iloc[0]) / 2
        else:
            half_w = pd.Timedelta(days=1)
        left  = res_date - half_w
        right = res_date + half_w
        return [left, right, right, left, left], [lo, lo, hi, hi, lo]

    # ── Static base traces ─────────────────────────────────────────────────────
    # Trace 0: pre-2025 history (never changes)
    # Traces 1–6: updated each frame
    base_data: list[go.BaseTraceType] = [
        go.Scatter(  # 0 — pre-simulation history
            x=pre_sim.index, y=pre_sim["price"],
            mode="lines",
            line=dict(color=CLR_PRE_SIM, width=1.5),
            name="WTI 2021–2024",
            showlegend=True,
        ),
        go.Scatter(  # 1 — revealed price (grows with simulation)
            x=[], y=[],
            mode="lines",
            line=dict(color=CLR_REVEALED, width=2.5),
            name="WTI Realized Price",
            showlegend=True,
        ),
        go.Scatter(  # 2 — accumulated past CI bars (no border — seamless fill)
            x=[], y=[],
            mode="lines",
            fill="toself",
            fillcolor=CLR_CI_PAST_FILL,
            line=dict(width=0),
            name="Past 95% CI",
            showlegend=True,
            hoverinfo="skip",
        ),
        go.Scatter(  # 3 — current forecast CI bar (not yet resolved)
            x=[], y=[],
            mode="lines",
            fill="toself",
            fillcolor=CLR_CI_CURR_FILL,
            line=dict(width=0),
            name="Current 95% CI (t+14)",
            showlegend=True,
            hoverinfo="skip",
        ),
        go.Scatter(  # 4 — current day vertical marker
            x=[], y=[],
            mode="lines",
            line=dict(color=CLR_DAY_LINE, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ),
        go.Scatter(  # 5 — resolution hits (inside CI)
            x=[], y=[],
            mode="markers",
            marker=dict(color=CLR_HIT, size=8, symbol="circle", opacity=0.9),
            name="Resolution: inside CI",
            showlegend=True,
        ),
        go.Scatter(  # 6 — resolution misses (outside CI)
            x=[], y=[],
            mode="markers",
            marker=dict(color=CLR_MISS, size=9, symbol="x", opacity=0.9),
            name="Resolution: outside CI",
            showlegend=True,
        ),
    ]

    fig = go.Figure(data=base_data)

    # ── Build frames ──────────────────────────────────────────────────────────
    frames: list[go.Frame] = []

    for sim_day in sim_days:
        revealed = price_df.loc[SIMULATION_START:sim_day]

        fc_rows = forecasts_df[forecasts_df["sim_day"] == sim_day]
        if fc_rows.empty:
            continue
        fc = fc_rows.iloc[0]

        past_res = forecasts_df[forecasts_df["resolution_date"] <= sim_day].dropna(subset=["actual_price"])
        hits     = past_res[past_res["inside_ci"]]
        misses   = past_res[~past_res["inside_ci"]]

        past_xs, past_ys   = _adjacent_bars(past_res)
        curr_xs, curr_ys   = _curr_bar_coords(fc, past_res)

        n_total = len(past_res)
        n_hits  = len(hits)
        pct     = (n_hits / n_total * 100) if n_total > 0 else 0.0

        scorecard = (
            f"<b>{sim_day.strftime('%b %d, %Y')}</b><br>"
            f"Resolutions: {n_total}<br>"
            f"In 95% CI: {n_hits}/{n_total} ({pct:.0f}%)"
        )

        frame = go.Frame(
            data=[
                go.Scatter(x=revealed.index.tolist(), y=revealed["price"].tolist()),          # 1
                go.Scatter(x=past_xs, y=past_ys),                                             # 2
                go.Scatter(x=curr_xs, y=curr_ys),                                             # 3
                go.Scatter(x=[sim_day, sim_day], y=[y_min, y_max]),                           # 4
                go.Scatter(                                                                    # 5
                    x=hits["resolution_date"].tolist(),
                    y=hits["actual_price"].tolist(),
                ),
                go.Scatter(                                                                    # 6
                    x=misses["resolution_date"].tolist(),
                    y=misses["actual_price"].tolist(),
                ),
            ],
            layout=go.Layout(
                annotations=[dict(
                    x=0.99, y=0.98,
                    xref="paper", yref="paper",
                    xanchor="right", yanchor="top",
                    text=scorecard,
                    showarrow=False,
                    font=dict(size=13, family="monospace"),
                    bgcolor="rgba(255,255,255,0.88)",
                    bordercolor="#bdbdbd",
                    borderwidth=1,
                    borderpad=8,
                    align="left",
                )]
            ),
            traces=[1, 2, 3, 4, 5, 6],
            name=str(sim_day.date()),
        )
        frames.append(frame)

    fig.frames = frames

    # ── Slider steps ──────────────────────────────────────────────────────────
    slider_steps = [
        dict(
            args=[[f.name], dict(frame=dict(duration=80, redraw=True), mode="immediate")],
            label=f.name if i % 20 == 0 else "",
            method="animate",
        )
        for i, f in enumerate(frames)
    ]

    # ── Play / Pause — placed to the left of the slider ──────────────────────
    updatemenus = [dict(
        type="buttons",
        showactive=False,
        direction="right",
        x=0.0,
        xanchor="left",
        y=-0.06,
        yanchor="top",
        pad=dict(r=4, t=0),
        buttons=[
            dict(
                label="▶  Play",
                method="animate",
                args=[None, dict(
                    frame=dict(duration=80, redraw=True),
                    fromcurrent=True,
                    transition=dict(duration=0),
                )],
            ),
            dict(
                label="⏸  Pause",
                method="animate",
                args=[[None], dict(
                    frame=dict(duration=0, redraw=False),
                    mode="immediate",
                    transition=dict(duration=0),
                )],
            ),
        ],
    )]

    x_end = (SIMULATION_END + timedelta(days=25)).strftime("%Y-%m-%d")
    fig.update_layout(
        title=dict(
            text="WTI Crude Oil — 14-Day Ahead Prophet Forecast: CI Bars at Resolution Dates",
            font=dict(size=17),
        ),
        xaxis=dict(
            title="Date",
            range=["2022-01-01", x_end],
            showgrid=True,
            gridcolor="#f0f0f0",
        ),
        yaxis=dict(
            title="Price (USD/bbl)",
            range=[y_min, y_max],
            showgrid=True,
            gridcolor="#f0f0f0",
        ),
        template="plotly_white",
        height=570,
        margin=dict(t=70, b=120, l=60, r=40),
        legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
        updatemenus=updatemenus,
        sliders=[dict(
            active=0,
            steps=slider_steps,
            # Start the slider track to the right of the Play/Pause buttons
            x=0.18,
            len=0.82,
            y=-0.06,
            currentvalue=dict(
                prefix="Day: ",
                visible=True,
                xanchor="center",
                font=dict(size=12),
            ),
            transition=dict(duration=0),
            pad=dict(t=28, b=8),
        )],
    )

    return fig


anim_fig = build_forecast_animation(price_df, forecasts_df)
anim_fig.show()

In [14]:
# Export as a standalone HTML file — open in any browser, no running kernel needed.
# Great for presentations, sharing, or embedding in a slide deck.
html_path = Path("oil_forecast_animation.html")
anim_fig.write_html(
    str(html_path),
    include_plotlyjs="cdn",
    full_html=True,
    auto_play=False,
)
print(f"Animation saved → {html_path.resolve()}")

Animation saved → /Users/ethanjackson/agentic-forecasting/playground/energy_case_study/notebooks/oil_forecast_animation.html


---

## Act 3 — The Punchline

Let's separate the 2025 backtest from the 2026 reality and look at what the numbers actually say.

In [15]:
def make_punchline_charts(forecasts_df: pd.DataFrame) -> None:
    """Two charts: (1) CI coverage by period, (2) forecast error distribution."""
    resolved = forecasts_df.dropna(subset=["actual_price"]).copy()
    resolved["period"] = resolved["sim_day"].apply(
        lambda d: "2025 Backtest" if d.year == 2025 else "Q1 2026 Reality"
    )
    resolved["error"] = resolved["actual_price"] - resolved["yhat"]
    resolved["abs_error"] = resolved["error"].abs()

    # ── Coverage bar chart ─────────────────────────────────────────────────────
    coverage = (
        resolved.groupby("period", sort=False)
        .agg(total=("inside_ci", "count"), inside=("inside_ci", "sum"))
        .assign(coverage_pct=lambda d: d["inside"] / d["total"] * 100)
        .reset_index()
    )
    coverage["order"] = coverage["period"].map({"2025 Backtest": 0, "Q1 2026 Reality": 1})
    coverage = coverage.sort_values("order")

    bar_colors = ["#2171b5" if p == "2025 Backtest" else "#de2d26" for p in coverage["period"]]

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=coverage["period"],
        y=coverage["coverage_pct"],
        marker_color=bar_colors,
        text=[f"{v:.1f}%" for v in coverage["coverage_pct"]],
        textposition="outside",
        textfont=dict(size=15),
        width=0.4,
    ))
    fig1.add_hline(
        y=95,
        line=dict(color="#636363", dash="dash", width=1.5),
        annotation_text=" Expected 95%",
        annotation_position="right",
        annotation_font=dict(size=11, color="#636363"),
    )
    fig1.update_layout(
        title=dict(text="Forecast Coverage: % of resolutions inside the 95% CI", font=dict(size=16)),
        yaxis=dict(title="Coverage (%)", range=[0, 108], showgrid=True, gridcolor="#f0f0f0"),
        xaxis=dict(title=""),
        template="plotly_white",
        height=380,
        margin=dict(t=60, b=40, l=60, r=40),
        showlegend=False,
    )
    fig1.show()

    # ── Error distribution violin ──────────────────────────────────────────────
    fig2 = go.Figure()
    period_order = ["2025 Backtest", "Q1 2026 Reality"]
    period_colors = {"2025 Backtest": "#2171b5", "Q1 2026 Reality": "#de2d26"}
    for period in period_order:
        subset = resolved[resolved["period"] == period]["error"]
        color = period_colors[period]
        fig2.add_trace(go.Violin(
            y=subset,
            name=period,
            box_visible=True,
            meanline_visible=True,
            fillcolor=color,
            opacity=0.6,
            line_color=color,
            points="outliers",
        ))

    fig2.add_hline(y=0, line=dict(color="#252525", width=1, dash="dot"))
    fig2.update_layout(
        title=dict(
            text="Forecast Error Distribution: Actual minus Forecast (USD/bbl)",
            font=dict(size=16),
        ),
        yaxis=dict(title="Error (USD/bbl)", showgrid=True, gridcolor="#f0f0f0", zeroline=False),
        xaxis=dict(title=""),
        template="plotly_white",
        height=420,
        margin=dict(t=60, b=40, l=60, r=40),
        violinmode="group",
    )
    fig2.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    summary = resolved.groupby("period").agg(
        n_forecasts=("sim_day", "count"),
        coverage_pct=("inside_ci", lambda x: f"{x.mean()*100:.1f}%"),
        mae=("abs_error", lambda x: f"${x.mean():.2f}"),
        median_abs_error=("abs_error", lambda x: f"${x.median():.2f}"),
        max_abs_error=("abs_error", lambda x: f"${x.max():.2f}"),
    ).loc[period_order]
    print("\nSummary:")
    print(summary.to_string())


make_punchline_charts(forecasts_df)


Summary:
                 n_forecasts coverage_pct     mae median_abs_error max_abs_error
period                                                                          
2025 Backtest            252        78.6%   $5.73            $4.99        $17.54
Q1 2026 Reality           71        42.3%  $20.44           $21.61        $51.61


### What happened?

Through 2025, Prophet's 95% CI caught about 3 out of 4 resolutions.
Not textbook calibration — oil is volatile, and the model was already showing some drift
as prices trended upward through mid-2025 — but workable for a real risk-management workflow.

Then came early 2026. Conflict escalation in the Persian Gulf — and mounting concern about
Strait of Hormuz access — drove WTI prices from the low-$70s to over $110/bbl
in a matter of weeks. The model, whose most recent monthly refit had trained on data through
February 2026 and was still pricing crude at $60–70, had no framework for
what it was seeing. By late March and early April 2026, it was missing by **$40–50 per barrel**.

The model wasn't *wrong in principle*. It correctly described the distribution of outcomes
that would have been reasonable given everything it had ever seen.
It just had no way of knowing what it didn't know.

> **A forecaster that backtests adequately is not the same as a forecaster that's robust to regime change.**

The question for the bootcamp: *what class of methods could do better — and specifically,
where can AI and agentic AI add value?*

---

## Act 4 — What Could Have Helped?

Prophet is a strong statistical baseline. But it's blind to the world outside the price series.
Here are four information sources — and four classes of method — that a more capable
forecaster could exploit.

In [16]:
def make_futures_curve_chart(price_df: pd.DataFrame) -> go.Figure | None:
    """Snapshot of the WTI futures term structure from nearby NYMEX contracts.

    Uses Yahoo Finance contract tickers (e.g. CLH26.NYM for March 2026).
    Skips gracefully if the tickers are unavailable or expired.
    """
    month_codes = ["F", "G", "H", "J", "K", "M", "N", "Q", "U", "V", "X", "Z"]
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    month_map = dict(zip(month_codes, month_names))

    today = datetime.date.today()
    tickers: list[str] = []
    labels: list[str] = []

    for delta_months in range(1, 10):
        # Step forward month by month
        m = today.month - 1 + delta_months
        year = today.year + m // 12
        month = m % 12 + 1
        code = month_codes[month - 1]
        yr2 = str(year)[-2:]
        ticker = f"CL{code}{yr2}.NYM"
        tickers.append(ticker)
        labels.append(f"{month_map[code]} '{yr2}")

    prices: list[float] = []
    valid_labels: list[str] = []

    for ticker, label in zip(tickers, labels):
        try:
            data = yf.download(ticker, period="5d", progress=False, auto_adjust=True)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            if not data.empty and "Close" in data.columns:
                val = float(data["Close"].dropna().iloc[-1])
                if val > 1.0:
                    prices.append(val)
                    valid_labels.append(label)
        except Exception:
            pass

    if not prices:
        print("⚠ Futures contract data not available — skipping term-structure chart.")
        return None

    spot = float(price_df["price"].iloc[-1])
    all_labels = ["Spot (now)"] + valid_labels
    all_prices = [spot] + prices

    contango = prices[-1] > prices[0] if len(prices) > 1 else False
    structure = "Contango — market prices in higher costs ahead" if contango else "Backwardation — market prices in near-term premium"
    curve_color = "#e6550d" if contango else "#2171b5"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=all_labels, y=all_prices,
        mode="lines+markers",
        line=dict(color=curve_color, width=2.5),
        marker=dict(size=9),
        name="WTI Futures Curve",
    ))
    fig.update_layout(
        title=dict(
            text=f"WTI Futures Term Structure — Current Snapshot<br><sup>{structure}</sup>",
            font=dict(size=16),
        ),
        xaxis=dict(title="Contract Month"),
        yaxis=dict(title="Price (USD/bbl)", showgrid=True, gridcolor="#f0f0f0"),
        template="plotly_white",
        height=380,
        margin=dict(t=80, b=50, l=60, r=40),
    )
    return fig


futures_fig = make_futures_curve_chart(price_df)
if futures_fig is not None:
    futures_fig.show()

### The Futures Curve

The futures term structure tells you what *the market* collectively believes about forward prices.
When the curve is in **backwardation** (near prices > far prices), traders are pricing a
near-term supply crunch but expecting relief later.
When it's in **contango** (near < far), the market sees near-term oversupply or weak demand.

Prophet can't see any of this — it only knows historical realized prices.
A futures-aware model can incorporate curve shape, spread dynamics, and roll signals as features.

But even futures markets can be caught off guard by a sudden geopolitical shock.

---

### Four Levers a Better Forecaster Could Pull

| Information source | What it adds | Limitation |
|---|---|---|
| **Futures curve** | Market-implied forward expectations; curve shape; spread signals | Still reactive — can be caught off guard by sudden shocks |
| **Prediction markets** | Probability-weighted crowd forecasts on discrete events (e.g. Strait of Hormuz closure) | Thin liquidity; slow to update on novel scenarios |
| **News & social signals** | Real-time event detection; geopolitical escalation indicators; sentiment | Noisy; requires reasoning to connect events to price impact |
| **Analyst scenarios & expert reasoning** | Structured scenario trees; domain expertise; conditional forecasts | Expensive, infrequent, not automated |

---

### Four Forecasting Method Families

| Method family | What it can do | What it can't do |
|---|---|---|
| **Statistical models** (Prophet, ARIMA, ETS) | Transparent; fast; well-calibrated on stable regimes | Blind to context; can't read the news |
| **ML / multivariate** (XGBoost, LightGBM, Ridge) | Incorporate engineered features: futures spreads, macro indicators | Still needs explicit feature engineering; can't reason |
| **Time-series foundation models** (Chronos, TimesFM, Moirai) | Pretrained on diverse series; zero-shot or fine-tuned generalization | New category — calibration and regime-break robustness still being evaluated |
| **LLM Processes + Agentic forecasters** | Retrieve news; reason through scenarios; call code; explain assumptions; update on new context in real time | Black-box risks; prompt sensitivity; require careful evaluation design |

---

### The Bootcamp Thesis

The hardest forecasting problems aren't the ones where the past predicts the future well.
They're the ones where **the world stops looking like it used to**.

In those moments, what matters most is:

1. **Awareness** — knowing that the regime has shifted
2. **Reasoning** — connecting external signals to a price view
3. **Uncertainty calibration** — widening the interval appropriately, not confidently predicting the wrong thing

Statistical models fail all three. Can AI agents do better?

**That's what we're here to find out.**

---

*Interested in participating? Reach out to the Vector AI Engineering team.*